# Graph-based multi-hop retrieval — tests

Same launcher shape as `colab_run_pipeline.ipynb`: clone the repo, install, run. `tests/README.md`
explains what each test is for; the short version:

| test | job |
|---|---|
| `test_pipeline_smoke.py` | Does the pipeline run end to end? 40 documents, 8 queries, scaled-down models, under a minute. Says nothing about graph quality. |
| `test_build_graphs.py` | Builds the four graph variants and dumps them to CSV so you can study them. Runs on the full corpus with the real models. |

**Before running:**
* Colab — *Runtime → Change runtime type → T4 GPU*. CPU works too, just slower.
* Kaggle — *Settings → Accelerator → GPU*, *Internet → On*.

Section 3 (pytest) is the quick pass/fail. Sections 4 and 5 run the same code with full logs
and write artifacts you can open.

## 1. Clone the repo

In [ ]:
REF = "main"  # branch for iteration, or a commit SHA to pin a run exactly
REPO_URL = "https://github.com/hadasy-tau/graphs_project.git"

import os

# Kaggle keeps writable state in /kaggle/working, Colab in /content, anywhere else: here.
BASE = next((d for d in ("/kaggle/working", "/content") if os.path.isdir(d)), os.getcwd())
REPO = os.path.join(BASE, "graphs_project")

if os.path.isdir(REPO):
    !cd {REPO} && git fetch --all --quiet && git checkout {REF} && git pull --ff-only || true
else:
    !git clone {REPO_URL} {REPO} && cd {REPO} && git checkout {REF}

os.chdir(REPO)  # every later cell, shell command included, runs from the repo root
!git log --oneline -1

## 2. Install dependencies

Both spaCy models: `en_core_web_sm` for `config/test_small.yaml` (the smoke test) and
`en_core_web_lg` for `config/base.yaml` (the graph construction test's default).

In [ ]:
# requirements.txt pins the en_core_web_lg 3.7.1 wheel. That pin drags spaCy back to 3.7.x
# and numpy below 2.0 with it, a downgrade the already-running kernel only picks up after a
# restart. So install everything else from the file and let spaCy fetch the model builds that
# match whatever version it resolved - same NER models, no downgrade, no restart.
lines = [l for l in open("requirements.txt").read().splitlines() if "en-core-web" not in l]
with open("/tmp/requirements-colab.txt", "w") as f:
    f.write("\n".join(lines) + "\n")

!pip install -q -r /tmp/requirements-colab.txt
!pip install -q pytest
!python -m spacy download en_core_web_sm
!python -m spacy download en_core_web_lg

In [ ]:
# Sanity check: fail here rather than halfway through a test.
import spacy
import torch

print("torch     :", torch.__version__)
print("GPU       :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE - the tests still run on CPU, just slower")

for model in ("en_core_web_sm", "en_core_web_lg"):
    spacy.load(model)
    print(f"spaCy NER : {model} OK")

try:
    import pcst_fast  # noqa: F401
    print("pcst_fast : OK (the real solver)")
except ImportError:
    print("pcst_fast : MISSING - tests/pcst_fallback.py substitutes a greedy heuristic, so the"
          " smoke test still runs. Fine for watching the flow, not for reporting numbers.")

## 3. pytest — both tests, pass/fail

The structural check: the smoke test end to end on the committed 40-document fixture, and
`test_build_graphs.py` on a 60-document slice with `config/test_small.yaml`. A couple of
minutes. The second one downloads MultiHop-RAG, so internet has to be on.

In [ ]:
!python -m pytest tests -v

## 4. Smoke test with the full log

Same test as above, run directly so its section banners print: `=== 2. preprocess ===` shows
what spaCy extracted, `=== 6. retrieval ===` compares gold against retrieved. Everything it
writes goes to `data/smoke_run/`.

In [ ]:
!python tests/test_pipeline_smoke.py

In [ ]:
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

display(pd.read_csv("data/smoke_run/metrics/summary_table.csv"))

## 5. Graph construction — the four variants as CSV

Defaults to the **full 609-document corpus with the real `base.yaml` models**, so these are
the graphs your experiments will use. Affordable because a graph's structure depends only on
the edge rules, so this skips the expensive part of `scripts/02` — embedding every edge text.

Set `DOCS` to iterate faster, but only the metadata graph survives subsetting exactly; the
entity and semantic rules are corpus-relative, so their edges shift. `tests/README.md` has
the table. The test prints which mode it ran in.

In [ ]:
CONFIG = "config/base.yaml"  # config/test_small.yaml for the scaled-down models
DOCS = None                  # None = the whole corpus; an int for a faster, approximate slice
QUERIES = None               # None = all of them (affects the connectivity report only)

args = f"--config {CONFIG}"
if DOCS:
    args += f" --docs {DOCS}"
if QUERIES:
    args += f" --queries {QUERIES}"

!python tests/test_build_graphs.py {args}

In [ ]:
from pathlib import Path

run_dir = max(Path("data/graph_runs").iterdir(), key=lambda p: p.stat().st_mtime)
print(f"run: {run_dir}\n")

# What each graph looks like...
display(pd.read_csv(run_dir / "graph_stats.csv"))

# ...and the question the project actually asks: can a graph reach each query's evidence?
# pairs_reachable below n_gold_pairs means the evidence sits in separate components, and no
# amount of graph walking will join it.
display(pd.read_csv(run_dir / "gold_connectivity.csv").head(20))

## 6. Download the artifacts

`data/graph_runs/` (edge lists, adjacency matrices, connectivity report) and `data/smoke_run/`
(the smoke test's graphs, retrieval output, and metrics), minus the cached `.pt` tensors.

In [ ]:
import shutil

BUNDLE = os.path.join(BASE, "graphs_project_test_output")
shutil.rmtree(BUNDLE, ignore_errors=True)

drop_tensors = shutil.ignore_patterns("*.pt")
shutil.copytree("data/graph_runs", os.path.join(BUNDLE, "graph_runs"), ignore=drop_tensors)
shutil.copytree("data/smoke_run", os.path.join(BUNDLE, "smoke_run"), ignore=drop_tensors)

zip_path = shutil.make_archive(BUNDLE, "zip", BUNDLE)
print(f"{zip_path}  ({os.path.getsize(zip_path) / 1e6:.1f} MB)")

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Kaggle: download it from the Output tab.")